In [1]:
"""
Step 1: Data Ingestion & Active-User Subsampling
---------------------------------------------------
Output: the FULL active-user cohort (K>=5 total events, 520,417 users) saved
as parquet files in data/subsampled/, for the rest of the team to build EDA
and modeling on top of, instead of everyone loading and cleaning the raw
files separately. No further random sampling is applied (SAMPLE_SIZE=None).

Chosen threshold: K>=5 total events (across purchase/add-to-cart/remove-from-cart).
Retains 19.4% of users but 70.0% of total event volume -- a deliberate tradeoff
to ensure users have enough history to form meaningful behavior sequences later. """

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [3]:
DATA_DIR = Path("Desktop/DS5500/data/raw")
OUTPUT_DIR = Path("Desktop/DS5500/data/subsampled")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "add_to_cart": "add_to_cart.parquet",
    "product_buy": "product_buy.parquet",
    "remove_from_cart": "remove_from_cart.parquet",
}

In [7]:
# Step 1a: Load, dedup, parse timestamps

raw = {}
for name, fname in FILES.items():
    df = pd.read_parquet(DATA_DIR / fname)
    df = df.drop_duplicates().copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    raw[name] = df
    print(f"{name}: {len(df):,} rows after dedup")

add_to_cart_clean = raw["add_to_cart"]
product_buy_clean = raw["product_buy"]
remove_from_cart_clean = raw["remove_from_cart"]

add_to_cart: 7,379,624 rows after dedup
product_buy: 1,937,599 rows after dedup
remove_from_cart: 2,535,306 rows after dedup


In [9]:
# Step 1b: Total events per user (ALL event types combined), used to define
# "active user": this is different from purchase count alone, since a user
# with only 1 purchase but lots of cart activity is still worth keeping.

all_events = pd.concat(
    [
        add_to_cart_clean[["client_id"]].assign(event_type="add_to_cart"),
        product_buy_clean[["client_id"]].assign(event_type="product_buy"),
        remove_from_cart_clean[["client_id"]].assign(event_type="remove_from_cart"),
    ],
    ignore_index=True,
)
 
events_per_user = all_events.groupby("client_id").size().rename("total_events")
print("\nTotal-event distribution across all users:")
print(events_per_user.describe())


Total-event distribution across all users:
count    2.681661e+06
mean     4.419846e+00
std      1.488060e+01
min      1.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      4.000000e+00
max      3.801000e+03
Name: total_events, dtype: float64


In [11]:
# Step 1c: Compare candidate active-user thresholds (evidence for choosing K)
# Result: K>=5 keeps 19.4% of users but 70.0% of total events -- chosen as
# the active-user threshold since it balances cohort size against ensuring
# users have enough history to form a meaningful behavior sequence.

candidate_thresholds = [2, 3, 5, 10]
for k in candidate_thresholds:
    active_mask = events_per_user >= k
    n_users = active_mask.sum()
    pct_users = n_users / len(events_per_user) * 100
    pct_events = events_per_user[active_mask].sum() / events_per_user.sum() * 100
    print(
        f"K>={k:>2}: {n_users:>9,} users kept ({pct_users:5.1f}% of users), "
        f"{pct_events:5.1f}% of total events retained"
    )

ACTIVE_USER_THRESHOLD = 5
active_users = set(events_per_user[events_per_user >= ACTIVE_USER_THRESHOLD].index)
print(f"\nActive users at K={ACTIVE_USER_THRESHOLD}: {len(active_users):,}")

K>= 2: 1,387,654 users kept ( 51.7% of users),  89.1% of total events retained
K>= 3:   893,055 users kept ( 33.3% of users),  80.7% of total events retained
K>= 5:   520,417 users kept ( 19.4% of users),  70.0% of total events retained
K>=10:   233,186 users kept (  8.7% of users),  54.3% of total events retained

Active users at K=5: 520,417


In [13]:
# Step 1d: Use the FULL active cohort (no further random sampling).
# SAMPLE_SIZE=None means: keep every user that passed the K>=5 threshold.

SAMPLE_SIZE = None  # set to an integer (e.g. 100_000) to subsample further
RANDOM_SEED = 42

if SAMPLE_SIZE is not None and len(active_users) > SAMPLE_SIZE:
    rng = np.random.default_rng(RANDOM_SEED)
    active_users_sampled = set(
        rng.choice(list(active_users), size=SAMPLE_SIZE, replace=False)
    )
else:
    active_users_sampled = active_users

print(f"Final cohort size: {len(active_users_sampled):,}")

Final cohort size: 520,417


In [15]:
# Step 1e: Filter all three event files to the sampled cohort and save.
# Everyone downstream should load from data/subsampled/, not the raw files.

subsampled = {
    "add_to_cart": add_to_cart_clean[add_to_cart_clean["client_id"].isin(active_users_sampled)],
    "product_buy": product_buy_clean[product_buy_clean["client_id"].isin(active_users_sampled)],
    "remove_from_cart": remove_from_cart_clean[remove_from_cart_clean["client_id"].isin(active_users_sampled)],
}

for name, df in subsampled.items():
    out_path = OUTPUT_DIR / f"{name}_subsampled.parquet"
    df.to_parquet(out_path, index=False)
    print(f"Saved {name}: {len(df):,} rows -> {out_path}")

pd.Series(sorted(active_users_sampled), name="client_id").to_csv(
    OUTPUT_DIR / "active_user_ids.csv", index=False
)
print(f"Saved active user ID list -> {OUTPUT_DIR / 'active_user_ids.csv'}")

Saved add_to_cart: 4,890,231 rows -> Desktop\DS5500\data\subsampled\add_to_cart_subsampled.parquet
Saved product_buy: 1,171,501 rows -> Desktop\DS5500\data\subsampled\product_buy_subsampled.parquet
Saved remove_from_cart: 2,229,315 rows -> Desktop\DS5500\data\subsampled\remove_from_cart_subsampled.parquet
Saved active user ID list -> Desktop\DS5500\data\subsampled\active_user_ids.csv


In [17]:
# Step 1f: Validation.
# With SAMPLE_SIZE=None, active_users_sampled == active_users, so this check
# is now a pure sanity check -- all differences should print as 0.00 pts.
# If you later set SAMPLE_SIZE to a number, this becomes a real check again:
# it validates the random draw against the active cohort it came from (NOT
# against the full unfiltered dataset, since the K>=5 filter is *designed*
# to look different from the full population by excluding low-activity users).

sub_buy = subsampled["product_buy"]
sub_cart = subsampled["add_to_cart"]

active_buy_full = product_buy_clean[product_buy_clean["client_id"].isin(active_users)]
active_cart_full = add_to_cart_clean[add_to_cart_clean["client_id"].isin(active_users)]

active_purchases_per_user = active_buy_full.groupby("client_id").size()
sub_purchases_per_user = sub_buy.groupby("client_id").size()

active_exactly_1 = (active_purchases_per_user == 1).mean() * 100
active_up_to_5 = (active_purchases_per_user <= 5).mean() * 100
sub_exactly_1 = (sub_purchases_per_user == 1).mean() * 100
sub_up_to_5 = (sub_purchases_per_user <= 5).mean() * 100

active_purchase_users = set(active_buy_full["client_id"].unique())
active_cart_users = set(active_cart_full["client_id"].unique())
active_cart_pct = len(active_purchase_users & active_cart_users) / len(active_purchase_users) * 100

sub_purchase_users = set(sub_buy["client_id"].unique())
sub_cart_users = set(sub_cart["client_id"].unique())
sub_cart_pct = len(sub_purchase_users & sub_cart_users) / len(sub_purchase_users) * 100

print("\nValidation: 50K SAMPLE vs. the 520K ACTIVE COHORT it was drawn from")
print(f"% with exactly 1 purchase: active-cohort={active_exactly_1:.2f}%  |  sample={sub_exactly_1:.2f}%")
print(f"% with <=5 purchases:      active-cohort={active_up_to_5:.2f}%  |  sample={sub_up_to_5:.2f}%")
print(f"% with cart history:       active-cohort={active_cart_pct:.2f}%  |  sample={sub_cart_pct:.2f}%")

checks = {
    "exactly-1-purchase %": abs(active_exactly_1 - sub_exactly_1),
    "<=5-purchases %": abs(active_up_to_5 - sub_up_to_5),
    "cart-history %": abs(active_cart_pct - sub_cart_pct),
}
print("\nDifference from active-cohort stats (<3 pts = OK):")
for name, diff in checks.items():
    flag = "OK" if diff < 3 else "CHECK THIS"
    print(f"  {name:<22}: {diff:5.2f} pts  [{flag}]")

print(
    "\nNote: this cohort is NOT representative of the full unfiltered dataset's "
    "purchase sparsity -- that's expected, since the K>=5 active-user filter "
    "deliberately excludes most low-activity users. State this explicitly as "
    "a known tradeoff in the methodology write-up."
)


Validation: 50K SAMPLE vs. the 520K ACTIVE COHORT it was drawn from
% with exactly 1 purchase: active-cohort=26.82%  |  sample=26.82%
% with <=5 purchases:      active-cohort=81.92%  |  sample=81.92%
% with cart history:       active-cohort=96.83%  |  sample=96.83%

Difference from active-cohort stats (<3 pts = OK):
  exactly-1-purchase %  :  0.00 pts  [OK]
  <=5-purchases %       :  0.00 pts  [OK]
  cart-history %        :  0.00 pts  [OK]

Note: this cohort is NOT representative of the full unfiltered dataset's purchase sparsity -- that's expected, since the K>=5 active-user filter deliberately excludes most low-activity users. State this explicitly as a known tradeoff in the methodology write-up.
